In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Classification Algorithms
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from whatsapp import WhatsappFilrt

# ==============================================================================
# 1. PREPARE DATASET
# ==============================================================================
wf = WhatsappFilrt()
df = wf.PreProcessing("Chat.txt")

# Clean inputs safely
X = df["Text"].fillna("").astype(str).str.lower().str.strip()
y = df["Name"].fillna("Unknown").astype(str).str.lower().str.strip()

# Drop blank rows/targets
valid_mask = (X != "") & (y != "unknown") & (y != "")
X, y = X[valid_mask], y[valid_mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==============================================================================
# 2. DEFINE VECTORIZERS & CLASSIFIERS
# ==============================================================================
vectorizers = {
    "CountVectorizer": CountVectorizer(ngram_range=(1, 2), min_df=2),
    "TF-IDF Vectorizer": TfidfVectorizer(ngram_range=(1, 2), min_df=2)
}

classifiers = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Complement Naive Bayes": ComplementNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0),
    "Linear Support Vector (SVC)": LinearSVC(dual='auto', C=1.0, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# ==============================================================================
# 3. TRAIN AND EVALUATE ALL COMBINATIONS
# ==============================================================================
results = []
best_model_pipe = None
best_f1_score = 0.0
best_model_name = ""

print("Starting model grid evaluation...\n" + "=" * 65)

for vec_name, vec_obj in vectorizers.items():
    for clf_name, clf_obj in classifiers.items():
        
        # Build combined classification pipeline
        pipeline = Pipeline([
            ('vectorizer', vec_obj),
            ('classifier', clf_obj)
        ])
        
        # Fit model on training data
        pipeline.fit(X_train, y_train)
        
        # Predict on test set
        y_pred = pipeline.predict(X_test)
        
        # Metrics computation
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')
        
        combo_title = f"{vec_name} + {clf_name}"
        
        results.append({
            "Vectorizer": vec_name,
            "Classifier": clf_name,
            "Accuracy": round(acc, 4),
            "Macro F1": round(f1_macro, 4),
            "Weighted F1": round(f1_weighted, 4)
        })
        
        # Track the top performing pipeline
        if f1_weighted > best_f1_score:
            best_f1_score = f1_weighted
            best_model_pipe = pipeline
            best_model_name = combo_title

# ==============================================================================
# 4. RESULTS COMPARISON TABLE
# ==============================================================================
results_df = pd.DataFrame(results).sort_values(by="Weighted F1", ascending=False)

print("\n=== CLASSIFICATION BENCHMARK SUMMARY ===")
print(results_df.to_string(index=False))
print("=" * 65)

# Save top performing model pipeline to disk
if best_model_pipe:
    saved_filename = "best_whatsapp_classifier.pkl"
    joblib.dump(best_model_pipe, saved_filename)
    print(f"\nBest Model: '{best_model_name}' (Weighted F1: {best_f1_score:.4f})")
    print(f"Saved top performing pipeline to '{saved_filename}'")

Starting model grid evaluation...

=== CLASSIFICATION BENCHMARK SUMMARY ===
       Vectorizer                  Classifier  Accuracy  Macro F1  Weighted F1
  CountVectorizer         Logistic Regression    0.3960    0.2402       0.3649
  CountVectorizer Linear Support Vector (SVC)    0.3812    0.2693       0.3533
TF-IDF Vectorizer Linear Support Vector (SVC)    0.3738    0.2373       0.3473
  CountVectorizer      Complement Naive Bayes    0.3589    0.2260       0.3446
TF-IDF Vectorizer               Random Forest    0.3688    0.2656       0.3438
  CountVectorizer     Multinomial Naive Bayes    0.3787    0.1901       0.3412
TF-IDF Vectorizer      Complement Naive Bayes    0.3515    0.2288       0.3400
TF-IDF Vectorizer         Logistic Regression    0.3812    0.1910       0.3375
  CountVectorizer               Random Forest    0.3614    0.2551       0.3374
TF-IDF Vectorizer     Multinomial Naive Bayes    0.3639    0.1552       0.3158

Best Model: 'CountVectorizer + Logistic Regression' (W